# PA2: 04_generate_persona

In [ ]:
import base64, io, zipfile, os, subprocess
from pathlib import Path
root = Path('/kaggle/working/pa2')
root.mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(root)
os.chdir(root)
subprocess.run(['python', '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch, transformers, datasets
print({'torch': torch.__version__, 'transformers': transformers.__version__, 'datasets': datasets.__version__})
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, 'Select GPU T4 x2'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2)), 'Expected two T4 GPUs'
subprocess.run(['python', '-m', 'pytest', '-q', 'tests'], check=True)


In [ ]:
matches=list(Path('/kaggle/input').rglob('persona/best/config.json'))
assert len(matches)==1
models=[('HuggingFaceTB/SmolLM2-360M-Instruct','smollm2-instruct'),(str(matches[0].parent),'persona_lora')]
prompts='eval_sets/persona_eval_seed42.json'
for model,name in models:
    subprocess.run(['python','-m','scripts.generate','--model',model,'--prompts',prompts,
        '--output',f'outputs/sanity_persona/{name}.json','--generator',name,'--limit','3','--chat-template'],check=True)
    print(Path(f'outputs/sanity_persona/{name}.json').read_text())
    subprocess.run(['python','-m','scripts.generate','--model',model,'--prompts',prompts,
        '--output',f'outputs/eval/persona/{name}.json','--generator',name,'--chat-template'],check=True)
subprocess.run(['python','-m','scripts.build_pairs','--baseline','outputs/eval/persona/smollm2-instruct.json',
    '--candidate','outputs/eval/persona/persona_lora.json','--output','outputs/eval/persona/pairs.jsonl'],check=True)
